In [ ]:
"""
   Below is the sample scripts to demonstrate the embedding and to find the most similar document to the 
   query uisng cosine similarity and euclidean distances.
"""


from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances
from langchain_huggingface import HuggingFaceEmbeddings


# Embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
#vector = embeddings.embed_query("Hello")

documents=["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."]

# Below is the list of embedded vectors
embedded_docs = embeddings.embed_documents(documents)
print("Embedded Vectors: ", embedded_docs)


my_query  = "Who is a president of USA?"
embedded_query = embeddings.embed_query(my_query)

print(cosine_similarity([embedded_query], embedded_docs))
print(euclidean_distances([embedded_query], embedded_docs))

In [ ]:
"""
       Below is the sample code to
          1) Store list of text to LangChain FAISS Vector Store and 
          2) Finding similar documents from the vector store to given query.
"""

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

"""
        384 is the dimension of the embedding vectors and index is the flat index which means 
        it will store the vectors in a flat structure and L2 is the distance metric which is 
        used to calculate the distance between the vectors.
"""

index = faiss.IndexFlatL2(384) 


"""

1) InMemoryDocstore : 
                      The InMemoryDocstore from langchain_community.docstore.in_memoryHolds store documents 
                      in active memory (RAM) using a standard Python dictionary format.
                      1) Key: The key is a unique identifier for each document. It is used to retrieve the
                              document from the docstore.
                      2) Value: The value is the Langchain document.

                      Example:
                                {
                                    "key_1": Document(page_content="...", metadata={...}),
                                    "key_2": Document(page_content="...", metadata={...})
                                }

2) index_to_docstore_id:

                      index_to_docstore_id is a Python dictionary that acts as a bridge 
                      between a FAISS vector index and the InMemoryDocstore

                      Example:

                                {
                                    0: "8a4f-2b9c",  # Maps Row 0 to its text ID
                                    1: "3d9e-7f1a"   # Maps Row 1 to its text ID
                                }

3) 
    Below is the example how documents are stores in vectorstore

    {
    # 1. FAISS Index (Conceptually an array/matrix of vector coordinates)
    "faiss_index_matrix": [
        [0.12, -0.43, 0.91, ...],  # Row 0
        [0.88, 0.15, -0.22, ...],  # Row 1
    ],

    # 2. The Bridge Dictionary
    "index_to_docstore_id": {
        0: "8a4f-2b9c",  # Maps Row 0 to its text ID
        1: "3d9e-7f1a"   # Maps Row 1 to its text ID
    },

    # 3. The Document Store Dictionary
    "InMemoryDocstore": {
        "8a4f-2b9c": Document(page_content="LangChain simplifies AI development.", metadata={"source": "readme.md"}),
        "3d9e-7f1a": Document(page_content="FAISS handles fast similarity searches.", metadata={"source": "docs.txt"})
    }


"""
vector_store = FAISS(
    embedding_function = embeddings,      
    index = index,                        
    docstore = InMemoryDocstore(),        
    index_to_docstore_id={}               
)

# Adding the documents to the vector store
vector_store.add_texts(["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."])


# Bridge dictionary to map the index of the FAISS index to the document IDs in the InMemoryDocstore
print("Bridge Dictionary: ", vector_store.index_to_docstore_id)

# Retrieving the document from the InMemoryDocstore using the FAISS index ID
faiss_index_id = 1
docstore_id  = vector_store.index_to_docstore_id[faiss_index_id]

# Retrieving the document from the InMemoryDocstore using the docstore ID
page_content = vector_store.docstore.search(docstore_id).page_content
metadata = vector_store.docstore.search(docstore_id).metadata

# retreiving embedded vector from the FAISS index using the FAISS index ID
embededed_vector = vector_store.index.reconstruct(faiss_index_id)
embedded_vector_size = vector_store.index.reconstruct(faiss_index_id).shape

print("FAISS Index ID: ", faiss_index_id)
print("Embedded Vector: ", embededed_vector)
print("Embedded Vector Size: ", embedded_vector_size)
print("Document Store ID: ", docstore_id)
print("Page Content: ", page_content)
print("Metadata: ", metadata)

# below is the example to find the top 1most similar document to the query using the vector store.
vector_store.similarity_search("Who is a president of USA?", k=1)

# below is the example to find the top 2 most similar document to the query using the vector store.
vector_store.similarity_search("Who is a president of USA?", k=2)

In [ ]:
"""
       Below is the sample code to
          1) Store Langchain documents to LangChain FAISS Vector Store. 
          2) Finding similar documents from the vector store to given query.
          3) Save the vector store to disk using save_local() method.
"""


from uuid import uuid4
from langchain_core.documents import Document
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


# Embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# faiss index
index = faiss.IndexFlatL2(384)

# vector store initializing
vector_store = FAISS(
    embedding_function = embeddings,      
    index = index,                        
    docstore = InMemoryDocstore(),        
    index_to_docstore_id={}               
)


document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1, document_2, document_3, document_4, document_5, document_6, document_7, document_8, document_9, document_10,
]

# Adding the documents to the vector store
vector_store.add_documents(documents=documents)

# to get top 5 most similar documents to the query from the vector store.
vector_store.similarity_search("LangChain provides abstractions to make working with LLMs easy", k=5)

# to get top 3 most similar documents to the query from the vector store with filter on metadata.
vector_store.similarity_search("LangChain provides abstractions to make working with LLMs easy", k=3, filter={"source": "news"})

"""
   1. LangChain's FAISS vector store provides a save_local() method to persist the vector store to disk, 
      allowing it to be reused later without recreating the index whenever application starts.
   2. The save_local() method saves two files in folder "faiss_index":
        a) .faiss file → Stores the FAISS index, which contains the embedded vectors and indexing structure.
        b) .pkl file → Stores the InMemoryDocstore and the index_to_docstore_id mapping, which links the 
            vectors in the FAISS index to their corresponding documents.
"""
vector_store.save_local("faiss_index")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2946.82it/s]


In [ ]:
"""
     Below is the code to 
         Load the persisted FAISS vector store from disk (which was stored by above script) using 
         load_local() method and use it for similarity search.
"""
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
vector_store.similarity_search("which is framework used to build NLP applications?", k=1)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4186.78it/s]


[Document(id='1a6444e6-92e0-4e20-846e-bdbf17907c23', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]